# Yelp AHA Review Analysis
* Input: facilities, categories, and reviews csv files
* Filter reviews by facility, category, keyword, date, ...
* Descriptive stats
* LDA

In [1]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel
from gensim.test.utils import datapath
import spacy
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

import pandas as pd
import math
import numpy as np
import scipy as sp

import re
import os
import sys
import datetime
import glob
from pprint import pprint

import pyLDAvis
import pyLDAvis.gensim
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()
%matplotlib inline

pd.set_option('display.max_columns',100)
pd.set_option('display.max_rows',200)
pd.set_option('display.precision',4)

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/arthurpelullo/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Program Parameters

### Constants

In [3]:
# Project data
PROJ_CODE = 'p03_AHA_hospitals'
FAC_LIST = True
FAC_LIST_FILE = 'aha_yelp_matched.csv'
FAC_MATCH_THRESH = 0.5
CAT_SRC = 'all'
REV_SRC = 'covid'
INF_REV_SRC = None

# Set date ranges: t1=t2=107 days
T1_START = pd.to_datetime('2019-11-30')
T1_END = pd.to_datetime('2020-03-15')
T1_RANGE = pd.date_range(start=T1_START, end=T1_END)

T2_START = pd.to_datetime('2020-03-16')
T2_END = pd.to_datetime('2020-06-30')
T2_RANGE = pd.date_range(start=T2_START, end=T2_END)

# Linguistic
WORD_COUNT = 20
NUM_TOPICS = 20
TOPIC_LABELS = ['Topic_'+str(item) for item in list(range(0,NUM_TOPICS))]

KEEP_WORDS = {''}
REMOVE_WORDS = {'from', 'subject', 're', 'edu', 'use'}
STOP_WORDS = stopwords.words('english')
STOP_WORDS.extend(REMOVE_WORDS)
NLP = spacy.load('en', disable=['parser', 'ner'])

# Plotting
NUM_COLS = {10:2,20:4,30:6}
HEIGHTS = {'large':[2000,2200,2400,2600,2800],'med':[200,220,240,260,280],'small':[10,12,14,16,18]}

# Misc.
MAX_ATTEMPTS = 100
MAX_CATS = 6

In [4]:
# Category Inclusion
CATS_REFINED = ['Surgeons','Diagnostic Services','Urgent Care','Health Coach','Ear Nose & Throat',
                'Rehabilitation Center','Medical Centers','Obstetricians & Gynecologists','Podiatrists',
                'Laser Eye Surgery/Lasik','Plastic Surgeons','Fertility','Oncologist','Addiction Medicine',
                'Childbirth Education','Pediatricians','Physical Therapy','Internal Medicine','Laboratory Testing',
                'Pain Management','Rheumatologists','Dermatologists','Nutritionists','Endocrinologists',
                'Occupational Therapy','Counseling & Mental Health','Ophthalmologists','Ultrasound Imaging Centers',
                'Vascular Medicine','Orthopedists','Lactation Services','Osteopathic Physicians','Speech Therapists',
                'Cardiologists','Midwives','Emergency Medicine','Doctors','Radiologists','Pulmonologist',
                'Gastroenterologist','Spine Surgeons','Nephrologists','Psychiatrists','Nurse Practitioner',
                'Skin Care','Hospitals','Sports Medicine','Walk-in Clinics','Anesthesiologists','Home Health Care',
                'Family Practice','Retina Specialists','Pharmacy','Drugstores','IV Hydration','Allergists',
                'Concierge Medicine','Psychologists','Prenatal/Perinatal Care','Reproductive Health Services',
                'Neurologist','Preventive Medicine','Sleep Specialists','Emergency Rooms','Diagnostic Imaging',
                'Cosmetic Surgeons','Urologists']
CATS_HOSP = ['Hospitals']

CAT_DICT = {'all':None, 'refined':CATS_REFINED, 'hospitals':CATS_HOSP}
CAT_LIST = CAT_DICT[CAT_SRC]

### Paths

In [5]:
#### Fixed Paths ####
MALLET_PATH = '/mallet-2.0.8/bin/mallet'

BASE_PATH = '/' # replace as needed
DATA_PATH = BASE_PATH + 'data/'
CODE_PATH = BASE_PATH + 'code/'
ANALYSIS_PATH = BASE_PATH + 'analysis/'

PROJ_PATH = ANALYSIS_PATH + PROJ_CODE + '/'
MASTER_DATA_PATH = PROJ_PATH + 'master_data/'
MODEL_PATH = PROJ_PATH + 'models/'
OUTPUT_PATH = PROJ_PATH + 'output/'

DESC_PATH = OUTPUT_PATH + 'descriptive/'
LDA_PATH = OUTPUT_PATH + 'lda/'

In [6]:
#### Dynamic Paths ####

# Topic Path
TOPIC_PATH = LDA_PATH + str(NUM_TOPICS) + '_topics/'
if not os.path.exists(TOPIC_PATH):
    os.mkdir(TOPIC_PATH)

# Result path
RESULT_PATH = TOPIC_PATH + REV_SRC + '/'
if not os.path.exists(RESULT_PATH):
    os.mkdir(RESULT_PATH)
    os.mkdir(RESULT_PATH + 'inference/')
    
# Inference path
if INF_REV_SRC is not None:
    INF_PATH = RESULT_PATH + 'inference/' + INF_REV_SRC + '/'
    if not os.path.exists(INF_PATH):
        os.mkdir(INF_PATH)

### Search Terms

In [7]:
# Covid
GENERAL_TERMS = ['COVID','COVID19','covid_19','COVIDー19','COVID2019','Covid19us','2019-nCoV','COVD19','CODVID19','Covid-19',
 'nCoV2019','covid2020','Covid19pandemic','covid19out','covidkindness','covidiot','covidiots','Knowcovid','SARSCoV2',
 'corona','coronavirus','coronovirus','coronaviruspandemic','Coronaeffect','coronavirususa','coronavirusoutbreak',
 'coronavirusupdate','Coronapocalypse','coronapocolypse','CoronaOutbreak','the \’rona','the roni','virus','Pandemic',
 'MyPandemicSurvivalPlan','PreventEpidemics','publichealth','flattenthecurve','Quarantine','Quarantinelife',
 'quarantineactivities','Quarentineandchill','QuarantineAndChill','SocialDistancing','socialdistancingnow',
 'selfisolating','HarmReduction','LockDownSA','lockdowneffect','lockdownextension','lockdowndiaries','Stayhome',
 'istayhome','istayathome','Stayathome','StayTheFHome','stayhomestaysafe','StayAtHomeSaveLives','stayhomesavelives',
 'iwillsurvivechallenge','StayAtHomeChallenge','ViewFromMyWindow','TogetherAtHome','Withme','Alonetogether',
 'inthistogether','untiltomorrow','ImDoingFineBecause','Staysafe','see10send10','seeapupsendapup','Safehands',
 'handwashing','Handwashing','washyourhands','workfromhome','Peoplehavetowork','essentialworkers',
 'ThanksHealthHeroes','healthcareheroes','GetMePPE','Mask','facemasks','Pdoh','Sdoh','hiap']
GENERAL_TERMS = [word.lower() for word in GENERAL_TERMS]

FOOD_TERMS = ['TooSmallToFail','SaveAmericanHospitality','SaveRestaurants','RestaurantRecovery',
              'ReliefForRestaurants','RallyForRestaurants','SupportLocalRestaurants','SupportLocal',
              'SupportLocalBusiness','TheGreatAmericanTakeout','CarryOut','OrderIn','CurbSide','CurbSidePickup',
              'DineLocal','StillOpen','WereOpen']
FOOD_TERMS = [word.lower() for word in FOOD_TERMS]

POLITICAL_TERMS = ['trumpownseverydeath','Trumpliedpeopledied','Trumpliesamericansdie','trumpliespeopledie',
                   'Wisconsinpandemicvoting','Trumpgenocide','trumppandemic','gopgenocide']
POLITICAL_TERMS = [word.lower() for word in POLITICAL_TERMS]

STIGMA_TERMS = ['HateIsAVirus','WashTheHate','RacismIsAVirus','IAmNotCOVID19']
STIGMA_TERMS = [word.lower() for word in STIGMA_TERMS]

CONSPIRACY_TERMS = ['filmyourhospital','filmyourhospitals','filmyourhospitalchallenge','emptyhospital','dempanic',
                    'plandemic','5gkills','5gconspiracy']
CONSPIRACY_TERMS = [word.lower() for word in CONSPIRACY_TERMS]

SEARCH_TERMS = GENERAL_TERMS+FOOD_TERMS+POLITICAL_TERMS+STIGMA_TERMS+CONSPIRACY_TERMS
SEARCH_TERMS_LOWER = [word.lower() for word in SEARCH_TERMS]
SEARCH_STRING = '|'.join(SEARCH_TERMS).lower()

# Telemedicine
TELE_TERMS = ['Telemedicine','Mobile phone','telehealth','Virtual visit','Remote monitor','Home monitor','Phone visit',
'Phone-based','Virtual doctor','Zoom','Bluejeans','Globalmeet','Video chat','Video conference',]
TELE_TERMS = [word.lower() for word in TELE_TERMS]
TELE_STRING = '|'.join(TELE_TERMS).lower()

# BLM
BLM_TERMS = ['Black Lives Matter','BLM','Race','Racism','Discriminate','Discrimination','Discriminated',
             'Discriminatory','Micro-aggression']
BLM_TERMS = [word.lower() for word in BLM_TERMS]
BLM_STRING = '|'.join(BLM_TERMS).lower()

## Function Definitions

In [8]:
# Define functions for tokenization, stopwords, bigrams, trigrams and lemmatization
def sent_to_words(sentences):
    for sentence in sentences:
        yield(simple_preprocess(str(sentence), deacc=True))  # deacc=True removes punctuations
        
def remove_stopwords(texts):
    return [[word for word in simple_preprocess(str(doc)) if word not in STOP_WORDS] for doc in texts]

def make_bigrams(texts,bigram_mod):
    return [bigram_mod[doc] for doc in texts]

def make_trigrams(texts,bigram_mod,trigram_mod):
    return [trigram_mod[bigram_mod[doc]] for doc in texts]

def lemmatization(texts, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV']):
    """https://spacy.io/api/annotation"""
    texts_out = []
    for sent in texts:
        doc = NLP(" ".join(sent)) 
        texts_out.append([token.lemma_ for token in doc if token.pos_ in allowed_postags])
    return texts_out

In [9]:
def get_corpus(data):
    # Split docs into words
    data_words = list(sent_to_words(data))
    # Create bigram and trigram models
    bigram = gensim.models.Phrases(data_words, min_count=5, threshold=100)
    bigram_mod = gensim.models.phrases.Phraser(bigram)
    trigram = gensim.models.Phrases(bigram[data_words], threshold=100)
    trigram_mod = gensim.models.phrases.Phraser(trigram)
    # Prepare analysis data
    data_words_nostops = remove_stopwords(data_words)
    data_words_bigrams = make_bigrams(data_words_nostops,bigram_mod)
    data_lemmatized = lemmatization(data_words_bigrams, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV'])

    # Format for LDA
    id2word = corpora.Dictionary(data_lemmatized)
    corpus = [id2word.doc2bow(text) for text in data_lemmatized]
    
    return data_lemmatized,id2word,corpus

In [10]:
def get_topic_terms(lda_model,num_topics,word_count):
    # Create dataframe with multiindex
    group_labels = ['word','value']
    col_labels = pd.MultiIndex.from_product([TOPIC_LABELS, group_labels])
    row_labels = ['Word_'+str(item) for item in list(range(1,word_count+1))]
    topic_term_top20 = pd.DataFrame(index=row_labels, columns=col_labels)

    # Get top topic words
    temp = lda_model.show_topics(num_topics=num_topics,num_words=word_count,formatted=False)
    for item in temp:
        topic_num = 'Topic_' + str(item[0])
        words,values = zip(*item[1])
        # Assign values
        topic_term_top20.loc[:,(topic_num,'word')] = words
        topic_term_top20.loc[:,(topic_num,'value')] = values
        
    return topic_term_top20

In [11]:
def format_topics_sentences(ldamodel, corpus, texts):
    # Init output
    sent_topics_df = pd.DataFrame()

    # Get main topic in each document
    for i, row in enumerate(ldamodel[corpus]):
        row = sorted(row, key=lambda x: (x[1]), reverse=True)
        # Get the Dominant topic, Perc Contribution and Keywords for each document
        for j, (topic_num, prop_topic) in enumerate(row):
            if j == 0:  # => dominant topic
                wp = ldamodel.show_topic(topic_num)
                topic_keywords = ", ".join([word for word, prop in wp])
                sent_topics_df = sent_topics_df.append(pd.Series([int(topic_num), round(prop_topic,4), topic_keywords]), ignore_index=True)
            else:
                break
    sent_topics_df.columns = ['Dominant_Topic', 'Perc_Contribution', 'Topic_Keywords']

    # Add original text to the end of the output
    contents = pd.Series(texts)
    sent_topics_df = pd.concat([sent_topics_df, contents], axis=1)
    return(sent_topics_df)


In [12]:
def get_time_series(dataframe,start_date,end_date,state=None):
    idx = []
    values = []
    date_range = pd.date_range(start=start_date, end=end_date)
    if state is not None:
        temp = dataframe[dataframe['state']==state]['rev_created_time'].value_counts().sort_index()
    else:
        temp = dataframe['rev_created_time'].value_counts().sort_index()
    temp = temp[temp.index > start_date]
    for dt in date_range:
        idx.append(dt)
        if dt in temp.index:
            values.append(temp.loc[dt])
        else:
            values.append(0)
    plot_data = pd.DataFrame(values, index=idx, columns=['Review Count'])
    return plot_data

## Read Data

In [13]:
# Get date string
count=0
use_date = datetime.datetime.now().date()
date_str_fmt = str(use_date)
date_str = date_str_fmt.replace('-','')

while not any(date_str in item for item in glob.glob(DATA_PATH + '*')) or count==MAX_ATTEMPTS:
    use_date = use_date - pd.to_timedelta(1, unit='days')
    date_str_fmt = str(use_date)
    date_str = date_str_fmt.replace('-','')
    
    count+=1
    if count == MAX_ATTEMPTS:
        print('Could not find any data!')
        break

In [16]:
# Base data
reviews = pd.read_csv(DATA_PATH + date_str + '_reviews.csv', index_col=0)
facilities = pd.read_csv(DATA_PATH + date_str + '_facilities.csv', index_col=0)
categories = pd.read_csv(DATA_PATH + date_str + '_categories.csv', index_col=0)

In [17]:
# clean data
reviews = reviews[~(reviews['fac_id'].isin(['HH N.','Suhas V.']))]
reviews = reviews.dropna(subset=['review','rev_rating'])
# Adjust columns
reviews['review'] = reviews['review'].fillna('')
reviews['review'] = reviews['review'].str.lower()
reviews['rev_rating'] = reviews['rev_rating'].astype('int32')
reviews['rev_created_time'] = pd.to_datetime(reviews['rev_created_time'])

In [15]:
print('Reviews:', len(reviews))
print('Facilities:', len(facilities))
print('Categories:', len(categories))

Reviews: 4869793
Facilities: 520469
Categories: 874879


## Master Data: Covid and Telemedicine

In [18]:
# Filter for included facilities
if FAC_LIST:
    fac_ids = pd.read_csv(DATA_PATH + FAC_LIST_FILE)
    fac_ids = fac_ids[fac_ids['best_match_score']>=FAC_MATCH_THRESH].fac_id
    facilities = facilities[facilities['fac_id'].isin(fac_ids)]

In [16]:
# Flatten cat data
cat_data = []
for idx,group in categories.groupby(['fac_id'])['title']:
    temp = list(group.values)
    for i in range(MAX_CATS-len(group)):
        temp.append('')
    temp.append(idx)
    cat_data.append(temp)
categories_flat = pd.DataFrame(cat_data, columns=['cat1','cat2','cat3','cat4','cat5','cat6','fac_id'])

In [17]:
# Merge datasets
fac_cat = facilities.merge(categories_flat, on='fac_id', how='inner')
all_data = reviews.merge(fac_cat, on='fac_id', how='inner')

if CAT_LIST is not None:
    print('Filtering by category..')
    all_data = all_data[(all_data['cat1'].isin(CAT_LIST)) |
                        (all_data['cat2'].isin(CAT_LIST)) |
                        (all_data['cat3'].isin(CAT_LIST)) |
                        (all_data['cat4'].isin(CAT_LIST)) |
                        (all_data['cat5'].isin(CAT_LIST)) |
                        (all_data['cat6'].isin(CAT_LIST))]

In [18]:
print('Reviews:', len(reviews))
print('Facilities:', len(facilities))
print('Categories:', len(categories))
print()
print('Filtered Reviews:', len(all_data))
print('Filtered Facilities:', len(all_data.fac_id.unique()))

Reviews: 4869793
Facilities: 520469
Categories: 874879

Filtered Reviews: 4869793
Filtered Facilities: 520469


In [ ]:
# Partition data by date range and keyword
all_data_t1 = all_data[all_data['rev_created_time'].isin(T1_RANGE)]
covid_data_t1 = all_data_t1[all_data_t1['review'].str.contains(SEARCH_STRING, regex=True)]
tele_data_t1 = all_data_t1[all_data_t1['review'].str.contains(TELE_STRING, regex=True)]
merge_data_t1 = covid_data_t1[covid_data_t1['review'].str.contains(TELE_STRING, regex=True)]

all_data_t2 = all_data[all_data['rev_created_time'].isin(T2_RANGE)]
covid_data_t2 = all_data_t2[all_data_t2['review'].str.contains(SEARCH_STRING, regex=True)]
tele_data_t2 = all_data_t2[all_data_t2['review'].str.contains(TELE_STRING, regex=True)]
merge_data_t2 = covid_data_t2[covid_data_t2['review'].str.contains(TELE_STRING, regex=True)]

In [ ]:
# Save to csv
all_data_t1.to_csv(MASTER_DATA_PATH + 'all_data_t1_' + str(datetime.datetime.now().date()).replace('-','') + '.csv')
covid_data_t1.to_csv(MASTER_DATA_PATH + 'covid_data_t1_' + str(datetime.datetime.now().date()).replace('-','') + '.csv')
tele_data_t1.to_csv(MASTER_DATA_PATH + 'tele_data_t1_' + str(datetime.datetime.now().date()).replace('-','') + '.csv')
merge_data_t1.to_csv(MASTER_DATA_PATH + 'merge_data_t1_' + str(datetime.datetime.now().date()).replace('-','') + '.csv')

all_data_t2.to_csv(MASTER_DATA_PATH + 'all_data_t2_' + str(datetime.datetime.now().date()).replace('-','') + '.csv')
covid_data_t2.to_csv(MASTER_DATA_PATH + 'covid_data_t2_' + str(datetime.datetime.now().date()).replace('-','') + '.csv')
tele_data_t2.to_csv(MASTER_DATA_PATH + 'tele_data_t2_' + str(datetime.datetime.now().date()).replace('-','') + '.csv')
merge_data_t2.to_csv(MASTER_DATA_PATH + 'merge_data_t2_' + str(datetime.datetime.now().date()).replace('-','') + '.csv')

#### Summary Stats

In [ ]:
summary_stats = [[len(all_data_t1),all_data_t1.rev_rating.mean(),len(all_data_t1.fac_id.unique()),all_data_t1.groupby(['fac_id']).tail(1).review_count.mean(), all_data_t1.groupby(['fac_id']).tail(1).fac_rating.mean(),
                  len(all_data_t2),all_data_t2.rev_rating.mean(),len(all_data_t2.fac_id.unique()),all_data_t2.groupby(['fac_id']).tail(1).review_count.mean(), all_data_t2.groupby(['fac_id']).tail(1).fac_rating.mean()],
                 [len(covid_data_t1),covid_data_t1.rev_rating.mean(),len(covid_data_t1.fac_id.unique()),covid_data_t1.groupby(['fac_id']).tail(1).review_count.mean(), covid_data_t1.groupby(['fac_id']).tail(1).fac_rating.mean(),
                  len(covid_data_t2),covid_data_t2.rev_rating.mean(),len(covid_data_t2.fac_id.unique()),covid_data_t2.groupby(['fac_id']).tail(1).review_count.mean(), covid_data_t2.groupby(['fac_id']).tail(1).fac_rating.mean()],
                 [len(tele_data_t1),tele_data_t1.rev_rating.mean(),len(tele_data_t1.fac_id.unique()),tele_data_t1.groupby(['fac_id']).tail(1).review_count.mean(), tele_data_t1.groupby(['fac_id']).tail(1).fac_rating.mean(),
                  len(tele_data_t2),tele_data_t2.rev_rating.mean(),len(tele_data_t2.fac_id.unique()),tele_data_t2.groupby(['fac_id']).tail(1).review_count.mean(), tele_data_t2.groupby(['fac_id']).tail(1).fac_rating.mean()],
                 [len(merge_data_t1),merge_data_t1.rev_rating.mean(),len(merge_data_t1.fac_id.unique()),merge_data_t1.groupby(['fac_id']).tail(1).review_count.mean(), merge_data_t1.groupby(['fac_id']).tail(1).fac_rating.mean(),
                  len(merge_data_t2),merge_data_t2.rev_rating.mean(),len(merge_data_t2.fac_id.unique()),merge_data_t2.groupby(['fac_id']).tail(1).review_count.mean(), merge_data_t2.groupby(['fac_id']).tail(1).fac_rating.mean()]]
summary_stats = pd.DataFrame(summary_stats, index=['all_data','covid_data','tele_data','merge_data'])
summary_stats.columns = pd.MultiIndex.from_product([['T1: 2020-01-01 to 2020-03-15','T2: 2020-03-16 to 2020-06-10'],['total_reviews','avg_review_rating','total_facilities','avg_fac_reviews','avg_fac_rating',]])
summary_stats.to_csv(DESC_PATH + 'summary_stats_' + str(datetime.datetime.now().date()).replace('-','') + '.csv')

In [ ]:
summary_stats

#### Categories

In [ ]:
# all
temp_all_data_t1 = all_data_t1.groupby(['fac_id']).tail(1)
temp_all_data_t2 = all_data_t2.groupby(['fac_id']).tail(1)

temp_cats_t1 = pd.concat([temp_all_data_t1.cat1,temp_all_data_t1.cat2,temp_all_data_t1.cat3,temp_all_data_t1.cat4,temp_all_data_t1.cat5,temp_all_data_t1.cat6])
temp_cats_t1 = pd.Series([item for item in temp_cats_t1.to_list() if item!=''])
temp_cats_t1 = zip(temp_cats_t1.value_counts()[0:150].index,temp_cats_t1.value_counts()[0:150].values)
temp_cats_t1 = pd.DataFrame(temp_cats_t1)
temp_cats_t1.columns = ['name_t1','count_t1']

temp_cats_t2 = pd.concat([temp_all_data_t2.cat1,temp_all_data_t2.cat2,temp_all_data_t2.cat3,temp_all_data_t2.cat4,temp_all_data_t2.cat5,temp_all_data_t2.cat6])
temp_cats_t2 = pd.Series([item for item in temp_cats_t2.to_list() if item!=''])
temp_cats_t2 = zip(temp_cats_t2.value_counts()[0:150].index,temp_cats_t2.value_counts()[0:150].values)
temp_cats_t2 = pd.DataFrame(temp_cats_t2)
temp_cats_t2.columns = ['name_t2','count_t2']

category_prevalence = pd.concat([temp_cats_t1,temp_cats_t2], axis=1)

In [ ]:
# covid
temp_covid_data_t1 = covid_data_t1.groupby(['fac_id']).tail(1)
temp_covid_data_t2 = covid_data_t2.groupby(['fac_id']).tail(1)

temp_cats_covid_t1 = pd.concat([temp_covid_data_t1.cat1,temp_covid_data_t1.cat2,temp_covid_data_t1.cat3,temp_covid_data_t1.cat4,temp_covid_data_t1.cat5,temp_covid_data_t1.cat6])
temp_cats_covid_t1 = pd.Series([item for item in temp_cats_covid_t1.to_list() if item!=''])
temp_cats_covid_t1 = zip(temp_cats_covid_t1.value_counts()[0:150].index,temp_cats_covid_t1.value_counts()[0:150].values)
temp_cats_covid_t1 = pd.DataFrame(temp_cats_covid_t1)
temp_cats_covid_t1.columns = ['name_covid_t1','count_covid_t1']

temp_cats_covid_t2 = pd.concat([temp_covid_data_t2.cat1,temp_covid_data_t2.cat2,temp_covid_data_t2.cat3,temp_covid_data_t2.cat4,temp_covid_data_t2.cat5,temp_covid_data_t2.cat6])
temp_cats_covid_t2 = pd.Series([item for item in temp_cats_covid_t2.to_list() if item!=''])
temp_cats_covid_t2 = zip(temp_cats_covid_t2.value_counts()[0:150].index,temp_cats_covid_t2.value_counts()[0:150].values)
temp_cats_covid_t2 = pd.DataFrame(temp_cats_covid_t2)
temp_cats_covid_t2.columns = ['name_covid_t2','count_covid_t2']

category_prevalence = pd.concat([category_prevalence,temp_cats_covid_t1,temp_cats_covid_t2], axis=1)

In [ ]:
# tele
temp_tele_data_t1 = tele_data_t1.groupby(['fac_id']).tail(1)
temp_tele_data_t2 = tele_data_t2.groupby(['fac_id']).tail(1)

temp_cats_tele_t1 = pd.concat([temp_tele_data_t1.cat1,temp_tele_data_t1.cat2,temp_tele_data_t1.cat3,temp_tele_data_t1.cat4,temp_tele_data_t1.cat5,temp_tele_data_t1.cat6])
temp_cats_tele_t1 = pd.Series([item for item in temp_cats_tele_t1.to_list() if item!=''])
temp_cats_tele_t1 = zip(temp_cats_tele_t1.value_counts()[0:150].index,temp_cats_tele_t1.value_counts()[0:150].values)
temp_cats_tele_t1 = pd.DataFrame(temp_cats_tele_t1)
temp_cats_tele_t1.columns = ['name_tele_t1','count_tele_t1']

temp_cats_tele_t2 = pd.concat([temp_tele_data_t2.cat1,temp_tele_data_t2.cat2,temp_tele_data_t2.cat3,temp_tele_data_t2.cat4,temp_tele_data_t2.cat5,temp_tele_data_t2.cat6])
temp_cats_tele_t2 = pd.Series([item for item in temp_cats_tele_t2.to_list() if item!=''])
temp_cats_tele_t2 = zip(temp_cats_tele_t2.value_counts()[0:150].index,temp_cats_tele_t2.value_counts()[0:150].values)
temp_cats_tele_t2 = pd.DataFrame(temp_cats_tele_t2)
temp_cats_tele_t2.columns = ['name_tele_t2','count_tele_t2']

category_prevalence = pd.concat([category_prevalence,temp_cats_tele_t1,temp_cats_tele_t2], axis=1)

In [ ]:
# merge
temp_merge_data_t1 = merge_data_t1.groupby(['fac_id']).tail(1)
temp_merge_data_t2 = merge_data_t2.groupby(['fac_id']).tail(1)

if len(temp_merge_data_t1) > 0:
    temp_cats_merge_t1 = pd.concat([temp_merge_data_t1.cat1,temp_merge_data_t1.cat2,temp_merge_data_t1.cat3,temp_merge_data_t1.cat4,temp_merge_data_t1.cat5,temp_merge_data_t1.cat6])
    temp_cats_merge_t1 = pd.Series([item for item in temp_cats_merge_t1.to_list() if item!=''])
    temp_cats_merge_t1 = zip(temp_cats_merge_t1.value_counts()[0:150].index,temp_cats_merge_t1.value_counts()[0:150].values)
    temp_cats_merge_t1 = pd.DataFrame(temp_cats_merge_t1)
    temp_cats_merge_t1.columns = ['name_merge_t1','count_merge_t1']
    category_prevalence = pd.concat([category_prevalence,temp_cats_merge_t1], axis=1)

if len(temp_merge_data_t2) > 0:
    temp_cats_merge_t2 = pd.concat([temp_merge_data_t2.cat1,temp_merge_data_t2.cat2,temp_merge_data_t2.cat3,temp_merge_data_t2.cat4,temp_merge_data_t2.cat5,temp_merge_data_t2.cat6])
    temp_cats_merge_t2 = pd.Series([item for item in temp_cats_merge_t2.to_list() if item!=''])
    temp_cats_merge_t2 = zip(temp_cats_merge_t2.value_counts()[0:150].index,temp_cats_merge_t2.value_counts()[0:150].values)
    temp_cats_merge_t2 = pd.DataFrame(temp_cats_merge_t2)
    temp_cats_merge_t2.columns = ['name_merge_t2','count_merge_t2']
    category_prevalence = pd.concat([category_prevalence,temp_cats_merge_t2], axis=1)

In [ ]:
# Save
category_prevalence.to_csv(DESC_PATH + 'cat_prev_' + str(datetime.datetime.now().date()).replace('-','') + '.csv')

In [ ]:
print(len(category_prevalence))
category_prevalence.head()

## Linguistic Features

### Doc-term Matrices and Word Frequencies

In [ ]:
# Constants
NGRAM_RANGE = (1,3)
MIN_DF = 0.005

In [ ]:
# All data
vec_all_t1 = CountVectorizer(lowercase=True,ngram_range=NGRAM_RANGE,min_df=MIN_DF,stop_words=STOP_WORDS)
docterm_all_t1 = vec_all_t1.fit_transform(all_data_t1.review)
docterm_all_t1 = pd.DataFrame(docterm_all_t1.todense(), columns=vec_all_t1.get_feature_names())
sorted_cols_all_t1 = docterm_all_t1.sum().sort_values(ascending=False).index.values
docterm_all_t1 = docterm_all_t1[sorted_cols_all_t1]
top20_all_t1 = pd.DataFrame(docterm_all_t1.sum()[0:WORD_COUNT].items(), columns=['word_all_t1','count_all_t1'])

vec_all_t2 = CountVectorizer(lowercase=True,ngram_range=NGRAM_RANGE,min_df=MIN_DF,stop_words=STOP_WORDS)
docterm_all_t2 = vec_all_t2.fit_transform(all_data_t2.review)
docterm_all_t2 = pd.DataFrame(docterm_all_t2.todense(), columns=vec_all_t2.get_feature_names())
sorted_cols_all_t2 = docterm_all_t2.sum().sort_values(ascending=False).index.values
docterm_all_t2 = docterm_all_t2[sorted_cols_all_t2]
top20_all_t2 = pd.DataFrame(docterm_all_t2.sum()[0:WORD_COUNT].items(), columns=['word_all_t2','count_all_t2'])

top20_df = pd.concat([top20_all_t1,top20_all_t2], axis=1)

In [ ]:
# covid data
vec_covid_t1 = CountVectorizer(lowercase=True,ngram_range=NGRAM_RANGE,min_df=MIN_DF,stop_words=STOP_WORDS)
docterm_covid_t1 = vec_covid_t1.fit_transform(covid_data_t1.review)
docterm_covid_t1 = pd.DataFrame(docterm_covid_t1.todense(), columns=vec_covid_t1.get_feature_names())
sorted_cols_covid_t1 = docterm_covid_t1.sum().sort_values(ascending=False).index.values
docterm_covid_t1 = docterm_covid_t1[sorted_cols_covid_t1]
top20_covid_t1 = pd.DataFrame(docterm_covid_t1.sum()[0:WORD_COUNT].items(), columns=['word_covid_t1','count_covid_t1'])

vec_covid_t2 = CountVectorizer(lowercase=True,ngram_range=NGRAM_RANGE,min_df=MIN_DF,stop_words=STOP_WORDS)
docterm_covid_t2 = vec_covid_t2.fit_transform(covid_data_t2.review)
docterm_covid_t2 = pd.DataFrame(docterm_covid_t2.todense(), columns=vec_covid_t2.get_feature_names())
sorted_cols_covid_t2 = docterm_covid_t2.sum().sort_values(ascending=False).index.values
docterm_covid_t2 = docterm_covid_t2[sorted_cols_covid_t2]
top20_covid_t2 = pd.DataFrame(docterm_covid_t2.sum()[0:WORD_COUNT].items(), columns=['word_covid_t2','count_covid_t2'])

top20_df = pd.concat([top20_df,top20_covid_t1,top20_covid_t2], axis=1)

In [ ]:
# tele data
vec_tele_t1 = CountVectorizer(lowercase=True,ngram_range=NGRAM_RANGE,min_df=MIN_DF,stop_words=STOP_WORDS)
docterm_tele_t1 = vec_tele_t1.fit_transform(tele_data_t1.review)
docterm_tele_t1 = pd.DataFrame(docterm_tele_t1.todense(), columns=vec_tele_t1.get_feature_names())
sorted_cols_tele_t1 = docterm_tele_t1.sum().sort_values(ascending=False).index.values
docterm_tele_t1 = docterm_tele_t1[sorted_cols_tele_t1]
top20_tele_t1 = pd.DataFrame(docterm_tele_t1.sum()[0:WORD_COUNT].items(), columns=['word_tele_t1','count_tele_t1'])

vec_tele_t2 = CountVectorizer(lowercase=True,ngram_range=NGRAM_RANGE,min_df=MIN_DF,stop_words=STOP_WORDS)
docterm_tele_t2 = vec_tele_t2.fit_transform(tele_data_t2.review)
docterm_tele_t2 = pd.DataFrame(docterm_tele_t2.todense(), columns=vec_tele_t2.get_feature_names())
sorted_cols_tele_t2 = docterm_tele_t2.sum().sort_values(ascending=False).index.values
docterm_tele_t2 = docterm_tele_t2[sorted_cols_tele_t2]
top20_tele_t2 = pd.DataFrame(docterm_tele_t2.sum()[0:WORD_COUNT].items(), columns=['word_tele_t2','count_tele_t2'])

top20_df = pd.concat([top20_df,top20_tele_t1,top20_tele_t2], axis=1)

In [ ]:
# merge data
if len(merge_data_t1) > 0:
    vec_merge_t1 = CountVectorizer(lowercase=True,ngram_range=NGRAM_RANGE,min_df=MIN_DF,stop_words=STOP_WORDS)
    docterm_merge_t1 = vec_merge_t1.fit_transform(merge_data_t1.review)
    docterm_merge_t1 = pd.DataFrame(docterm_merge_t1.todense(), columns=vec_merge_t1.get_feature_names())
    sorted_cols_merge_t1 = docterm_merge_t1.sum().sort_values(ascending=False).index.values
    docterm_merge_t1 = docterm_merge_t1[sorted_cols_merge_t1]
    top20_merge_t1 = pd.DataFrame(docterm_merge_t1.sum()[0:WORD_COUNT].items(), columns=['word_merge_t1','count_merge_t1'])
    top20_df = pd.concat([top20_df,top20_merge_t1], axis=1)

if len(merge_data_t2) > 0:
    vec_merge_t2 = CountVectorizer(lowercase=True,ngram_range=NGRAM_RANGE,min_df=MIN_DF,stop_words=STOP_WORDS)
    docterm_merge_t2 = vec_merge_t2.fit_transform(merge_data_t2.review)
    docterm_merge_t2 = pd.DataFrame(docterm_merge_t2.todense(), columns=vec_merge_t2.get_feature_names())
    sorted_cols_merge_t2 = docterm_merge_t2.sum().sort_values(ascending=False).index.values
    docterm_merge_t2 = docterm_merge_t2[sorted_cols_merge_t2]
    top20_merge_t2 = pd.DataFrame(docterm_merge_t2.sum()[0:WORD_COUNT].items(), columns=['word_merge_t2','count_merge_t2'])
    top20_df = pd.concat([top20_df,top20_merge_t2], axis=1)

In [ ]:
# Save
top20_df.to_csv(DESC_PATH + 'word_freq_' + str(datetime.datetime.now().date()).replace('-','') + '.csv')

In [ ]:
top20_df.head()

### Topic Modeling

#### Preprocess data

In [ ]:
# Select raw data
if REV_SRC =='all':
    data_t1 = all_data_t1
    data_t2 = all_data_t2
if REV_SRC =='covid':
    data_t1 = covid_data_t1
    data_t2 = covid_data_t2
if REV_SRC =='tele':
    data_t1 = tele_data_t1
    data_t2 = tele_data_t2
    
data = pd.concat([data_t1,data_t2]).review.values.tolist()

In [ ]:
data_lemmatized,id2word,corpus = get_corpus(data)

In [ ]:
print('Reviews T1:',len(data_t1))
print('Reviews T2:',len(data_t2))
print('All Reviews:',len(data))

#### Mallet Model

In [ ]:
# Mallet topic model
ldamallet = gensim.models.wrappers.LdaMallet(MALLET_PATH,corpus=corpus,num_topics=NUM_TOPICS,id2word=id2word)
ldamalletConvert = gensim.models.wrappers.ldamallet.malletmodel2ldamodel(ldamallet, gamma_threshold=0.001, iterations=50)

In [ ]:
# Show topics and coherence score
pprint(ldamallet.show_topics(num_topics=NUM_TOPICS,num_words=WORD_COUNT,formatted=False))
coherence_model_ldamallet = CoherenceModel(model=ldamallet, texts=data_lemmatized, dictionary=id2word, coherence='c_v')
coherence_ldamallet = coherence_model_ldamallet.get_coherence()
print('\nCoherence Score: ', coherence_ldamallet)

#### Model Output

In [ ]:
# Doc-topic
doc_ldamallet = ldamallet[corpus]
doc_ldamallet_t1 = doc_ldamallet[0:len(data_t1)]
doc_ldamallet_t2 = doc_ldamallet[len(data_t1):]

doc_topic = pd.DataFrame([[topic[1] for topic in doc] for doc in doc_ldamallet], columns=TOPIC_LABELS)
doc_topic_t1 = pd.DataFrame([[topic[1] for topic in doc] for doc in doc_ldamallet_t1], columns=TOPIC_LABELS)
doc_topic_t2 = pd.DataFrame([[topic[1] for topic in doc] for doc in doc_ldamallet_t2], columns=TOPIC_LABELS)

doc_topic.to_csv(RESULT_PATH + 'doc_topic_' + REV_SRC + '_' + str(datetime.datetime.now().date()).replace('-','') + '.csv')

In [ ]:
doc_topic.head(5)

In [ ]:
# Topic-term
topic_term_ldamallet_top20 = get_topic_terms(ldamallet,num_topics=NUM_TOPICS,word_count=WORD_COUNT)
topic_term_ldamallet_top20.to_csv(RESULT_PATH + 'topic_term_' + REV_SRC + '_' + str(datetime.datetime.now().date()).replace('-','') + '.csv')

In [ ]:
topic_term_ldamallet_top20.round(3).head(5)

In [ ]:
# Visualize the topics
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(ldamalletConvert, corpus, id2word)
vis

#### Save models

In [ ]:
# Save models to disk
ldamallet_file = datapath(MODEL_PATH + 'ldamallet_' + REV_SRC + '_' + str(NUM_TOPICS)+ 'topics_' + str(datetime.datetime.now().date()).replace('-',''))
ldamallet.save(ldamallet_file)
ldamalletConvert_file = datapath(MODEL_PATH + 'ldamalletConvert_' + REV_SRC + '_' + str(NUM_TOPICS)+ 'topics_' + str(datetime.datetime.now().date()).replace('-',''))
ldamalletConvert.save(ldamalletConvert_file)

### Topic Analysis

#### Dominant Topic in each Document

In [ ]:
df_topic_sents_keywords = format_topics_sentences(ldamodel=ldamallet, corpus=corpus, texts=data)
dominant_topic = df_topic_sents_keywords.reset_index()
dominant_topic.columns = ['doc_num', 'dominant_topic', 'topic_contribution', 'topic_keywords', 'doc_text']
dominant_topic.to_csv(RESULT_PATH + 'doc_dominantTopic_' + REV_SRC + '_' + str(datetime.datetime.now().date()).replace('-','') + '.csv')

In [ ]:
print(len(dominant_topic))
dominant_topic.head(20)

#### Most Representative Document for each Topic

In [ ]:
group_data = []
for idx,group in dominant_topic.groupby(['dominant_topic']):
    group_data.append(group.sort_values(['topic_contribution'], ascending=False).iloc[0,:])
dominant_topic_rep = pd.DataFrame(group_data).reset_index(drop=True)
dominant_topic_rep.to_csv(RESULT_PATH + 'dominantTopic_repDoc_' + REV_SRC+'_'+str(datetime.datetime.now().date()).replace('-','')+'.csv')

In [ ]:
print(len(dominant_topic_rep))
dominant_topic_rep.head(20)

#### Dominant Topic Frequencies

In [ ]:
dominant_topic_t1 = dominant_topic[0:len(data_t1)]
dominant_topic_t2 = dominant_topic[len(data_t1):]

temp = pd.concat([dominant_topic['dominant_topic'].value_counts(0), dominant_topic['dominant_topic'].value_counts(1),
                  dominant_topic_t1['dominant_topic'].value_counts(0), dominant_topic_t1['dominant_topic'].value_counts(1),
                  dominant_topic_t2['dominant_topic'].value_counts(0), dominant_topic_t2['dominant_topic'].value_counts(1)], axis=1)
temp.columns = ['doc_count_all','doc_pct_all','doc_count_t1','doc_pct_t1','doc_count_t2','doc_pct_t2']
temp['doc_count_diff'] = temp['doc_count_t2'] - temp['doc_count_t1']
temp['doc_pct_diff'] = ((temp['doc_pct_t2'] - temp['doc_pct_t1'])/temp['doc_pct_t1'])*100

topic_freq = dominant_topic_rep[['dominant_topic','topic_keywords']].merge(temp, how='inner',left_index=True,right_index=True)
topic_freq.to_csv(RESULT_PATH + 'dominantTopic_freq_' + REV_SRC + '_' + str(datetime.datetime.now().date()).replace('-','') + '.csv')

In [ ]:
print(len(topic_freq))
topic_freq.head(20)

#### Topic Prevalence

In [ ]:
topic_prev = pd.concat([doc_topic.mean(),doc_topic_t1.mean(),doc_topic_t2.mean()], axis=1)
topic_prev.columns = ['topic_prev_all','topic_prev_t1','topic_prev_t2']
topic_prev['topic_prev_pct_diff'] = ((topic_prev['topic_prev_t2'] - topic_prev['topic_prev_t1'])/topic_prev['topic_prev_t1'])*100
topic_prev.to_csv(RESULT_PATH + 'topic_prev_' + REV_SRC + '_' + str(datetime.datetime.now().date()).replace('-','') + '.csv')

In [ ]:
print(len(topic_prev))
topic_prev.head(20)

#### Word Count Histogram

In [ ]:
doc_lens = [len(d.split()) for d in dominant_topic.doc_text]

# Plot
fig,ax = plt.subplots(1,1,figsize=(16,8))
ax.hist(doc_lens, bins = 200, color='navy')
ax.text(max(doc_lens)/2,HEIGHTS['small'][4],"Mean: " + str(round(np.mean(doc_lens))), fontsize=14)
ax.text(max(doc_lens)/2,HEIGHTS['small'][3],"Median: " + str(round(np.median(doc_lens))), fontsize=14)
ax.text(max(doc_lens)/2,HEIGHTS['small'][2],"Stdev: " + str(round(np.std(doc_lens))), fontsize=14)
ax.text(max(doc_lens)/2,HEIGHTS['small'][1],"1%: " + str(round(np.quantile(doc_lens, q=0.01))), fontsize=14)
ax.text(max(doc_lens)/2,HEIGHTS['small'][0],"99%: " + str(round(np.quantile(doc_lens, q=0.99))), fontsize=14)

ax.set_xlim=(0, max(doc_lens))
ax.set_ylabel('Number of Documents')
ax.set_xlabel('Document Word Count')
ax.tick_params(size=16)
ax.set_xticks(np.linspace(0,max(doc_lens),9))
ax.set_title('Distribution of Document Word Counts', fontdict=dict(size=22))
#plt.show()

name = RESULT_PATH + 'wordCount_hist_' + REV_SRC + '_' + str(datetime.datetime.now().date()).replace('-','') + '.png'
plt.savefig(name,bbox_inches='tight')

#### Word Count Histogram by Topic

In [ ]:
import seaborn as sns
import matplotlib.colors as mcolors
cols = [color for name, color in mcolors.XKCD_COLORS.items()]  # more colors: 'mcolors.XKCD_COLORS'

fig, axes = plt.subplots(5,NUM_COLS[NUM_TOPICS],figsize=(16,20), sharex=True, sharey=True)

for i, ax in enumerate(axes.flatten()):    
    dominant_topic_sub = dominant_topic.loc[dominant_topic.dominant_topic == i, :]
    doc_lens = [len(d.split()) for d in dominant_topic_sub.doc_text]
    ax.hist(doc_lens, bins = 50, color=cols[i])
    ax.tick_params(axis='y', labelcolor=cols[i], color=cols[i])
    sns.kdeplot(doc_lens, color="black", shade=False, ax=ax.twinx())
    ax.set(xlim=(0, 1000), xlabel='Document Word Count')
    ax.set_ylabel('Number of Documents', color=cols[i])
    ax.set_title('Topic: '+str(i), fontdict=dict(size=16, color=cols[i]))

fig.tight_layout()
fig.subplots_adjust(top=0.90)
plt.xticks(np.linspace(0,1000,9))
fig.suptitle('Distribution of Document Word Counts by Dominant Topic', fontsize=22)
plt.show()

name = RESULT_PATH + 'wordCount_hist_topics_' + REV_SRC + '_' + str(datetime.datetime.now().date()).replace('-','') + '.png'
fig.savefig(name,bbox_inches='tight')

#### Word Clouds

In [ ]:
# 1. Wordcloud of Top N words in each topic
from matplotlib import pyplot as plt
from wordcloud import WordCloud, STOPWORDS
import matplotlib.colors as mcolors

cols = [color for name, color in mcolors.XKCD_COLORS.items()]  # more colors: 'mcolors.XKCD_COLORS'

cloud = WordCloud(stopwords=STOP_WORDS,
                  background_color='white',
                  width=2500,
                  height=1800,
                  max_words=WORD_COUNT,
                  colormap='tab10',
                  color_func=lambda *args, **kwargs: cols[i],
                  prefer_horizontal=1.0)

topics = ldamallet.show_topics(formatted=False,num_topics=NUM_TOPICS,num_words=WORD_COUNT)

fig, axes = plt.subplots(5,NUM_COLS[NUM_TOPICS], figsize=(16,20), sharex=True, sharey=True)

for i, ax in enumerate(axes.flatten()):
    fig.add_subplot(ax)
    topic_words = dict(topics[i][1])
    cloud.generate_from_frequencies(topic_words, max_font_size=350)
    plt.gca().imshow(cloud)
    plt.gca().set_title('Topic ' + str(i), fontdict=dict(size=16))
    plt.gca().axis('off')


plt.subplots_adjust(wspace=0, hspace=0)
plt.axis('off')
plt.margins(x=0, y=0)
plt.tight_layout()
plt.show()

name = RESULT_PATH + 'wordCloud_topics_' + REV_SRC + '_' + str(datetime.datetime.now().date()).replace('-','') + '.png'
fig.savefig(name,bbox_inches='tight')

#### Word Counts by Topic

In [ ]:
from collections import Counter
topics = ldamallet.show_topics(formatted=False,num_topics=NUM_TOPICS)
data_flat = [w for w_list in data_lemmatized for w in w_list]
counter = Counter(data_flat)

out = []
for i, topic in topics:
    for word, weight in topic:
        out.append([word, i , weight, counter[word]])

df = pd.DataFrame(out, columns=['word', 'topic_id', 'importance', 'word_count'])        

# Plot Word Count and Weights of Topic Keywords
fig, axes = plt.subplots(5,NUM_COLS[NUM_TOPICS], figsize=(20,20), sharey=True)
cols = [color for name, color in mcolors.XKCD_COLORS.items()]
for i, ax in enumerate(axes.flatten()):
    ax.bar(x='word', height="word_count", data=df.loc[df.topic_id==i, :], color=cols[i], width=0.5, alpha=0.3, label='Word Count')
    ax_twin = ax.twinx()
    ax_twin.bar(x='word', height="importance", data=df.loc[df.topic_id==i, :], color=cols[i], width=0.2, label='Weights')
    ax.set_ylabel('Word Count', color=cols[i])
    ax_twin.set_ylim(0, 0.060); ax.set_ylim(0, 10000)
    ax.set_title('Topic: ' + str(i), color=cols[i], fontsize=16)
    ax.tick_params(axis='y', left=False)
    ax.set_xticklabels(df.loc[df.topic_id==i, 'word'], rotation=30, horizontalalignment= 'right')
    ax.legend(loc='upper left'); ax_twin.legend(loc='upper right')

fig.tight_layout(w_pad=2)    
fig.suptitle('Word Count and Importance of Topic Keywords', fontsize=22, y=1.05)
plt.show()

name = RESULT_PATH + 'wordCount_terms_topics_' + REV_SRC + '_' + str(datetime.datetime.now().date()).replace('-','') + '.png'
fig.savefig(name,bbox_inches='tight')

#### Topic Clusters

In [ ]:
# Get topic weights and dominant topics ------------
from sklearn.manifold import TSNE
from bokeh.plotting import figure, output_file, show
from bokeh.models import Label
from bokeh.io import output_notebook
from bokeh.io import export_png

# Get topic weights
topic_weights = []
for i, row_list in enumerate(ldamallet[corpus]):
    topic_weights.append([w for i, w in row_list])

# Array of topic weights    
arr = pd.DataFrame(topic_weights).fillna(0).values

# Keep the well separated points (optional)
arr = arr[np.amax(arr, axis=1) > 0.1]

# Dominant topic number in each doc
topic_num = np.argmax(arr, axis=1)

# tSNE Dimension Reduction
tsne_model = TSNE(n_components=2, verbose=1, random_state=0, angle=.99, init='pca')
tsne_lda = tsne_model.fit_transform(arr)

# Plot the Topic Clusters using Bokeh
output_notebook()
n_topics = NUM_TOPICS
mycolors = np.array([color for name, color in mcolors.XKCD_COLORS.items()])
plot = figure(title="t-SNE Clustering of {} LDA Topics".format(n_topics), 
              plot_width=900, plot_height=700)
plot.scatter(x=tsne_lda[:,0], y=tsne_lda[:,1], color=mycolors[topic_num])
show(plot)

name = RESULT_PATH + 'doc_topic_clusters_' + REV_SRC + '_' + str(datetime.datetime.now().date()).replace('-','') + '.png'
export_png(plot, filename=name)

### Inference

In [ ]:
# Select raw data
if INF_REV_SRC =='covid':
    inf_data_t1 = covid_data_t1
    inf_data_t2 = covid_data_t2
if INF_REV_SRC =='tele':
    inf_data_t1 = tele_data_t1
    inf_data_t2 = tele_data_t2
if INF_REV_SRC =='merge':
    inf_data_t1 = merge_data_t1
    inf_data_t2 = merge_data_t2
    
inf_data = pd.concat([inf_data_t1,inf_data_t2]).review.values.tolist()

In [ ]:
inf_lemma,inf_id2word,inf_corpus = get_corpus(inf_data)

In [ ]:
print('Reviews T1:',len(inf_data_t1))
print('Reviews T2:',len(inf_data_t2))
print('All Reviews:',len(inf_data))

In [ ]:
inf_doc_ldamallet = ldamallet[inf_corpus]
inf_doc_ldamallet_t1 = inf_doc_ldamallet[0:len(inf_data_t1)]
inf_doc_ldamallet_t2 = inf_doc_ldamallet[len(inf_data_t1):]

inf_doc_topic = pd.DataFrame([[topic[1] for topic in doc] for doc in inf_doc_ldamallet], columns=TOPIC_LABELS)
inf_doc_topic_t1 = pd.DataFrame([[topic[1] for topic in doc] for doc in inf_doc_ldamallet_t1], columns=TOPIC_LABELS)
inf_doc_topic_t2 = pd.DataFrame([[topic[1] for topic in doc] for doc in inf_doc_ldamallet_t2], columns=TOPIC_LABELS)

inf_doc_topic.to_csv(INF_PATH + 'doc_topic_' + INF_REV_SRC + '_' + str(datetime.datetime.now().date()).replace('-','') + '.csv')

In [ ]:
print(len(inf_doc_topic))
inf_doc_topic.head(20)

#### Dominant Topic in each Document

In [ ]:
inf_dominant_topic_temp = format_topics_sentences(ldamodel=ldamallet, corpus=inf_corpus, texts=inf_data)
inf_dominant_topic = inf_dominant_topic_temp.reset_index()
inf_dominant_topic.columns = ['doc_num', 'dominant_topic', 'topic_contribution', 'topic_keywords', 'doc_text']
inf_dominant_topic.to_csv(INF_PATH + 'doc_dominantTopic_' + INF_REV_SRC + '_' + str(datetime.datetime.now().date()).replace('-','') + '.csv')

In [ ]:
print(len(inf_dominant_topic))
inf_dominant_topic.head(20)

#### Most Representative Document for each Topic

In [ ]:
group_data = []
for idx,group in inf_dominant_topic.groupby(['dominant_topic']):
    group_data.append(group.sort_values(['topic_contribution'], ascending=False).iloc[0,:])
inf_dominant_topic_rep = pd.DataFrame(group_data).reset_index(drop=True)
inf_dominant_topic_rep.to_csv(INF_PATH + 'dominantTopic_repDoc_' + INF_REV_SRC + '_' + str(datetime.datetime.now().date()).replace('-','') + '.csv')

In [ ]:
print(len(inf_dominant_topic_rep))
inf_dominant_topic_rep.head(20)

#### Dominant Topic Frequencies

In [ ]:
inf_dominant_topic_t1 = inf_dominant_topic[0:len(inf_data_t1)]
inf_dominant_topic_t2 = inf_dominant_topic[len(inf_data_t1):]

temp = pd.concat([inf_dominant_topic['dominant_topic'].value_counts(0), inf_dominant_topic['dominant_topic'].value_counts(1),
                  inf_dominant_topic_t1['dominant_topic'].value_counts(0), inf_dominant_topic_t1['dominant_topic'].value_counts(1),
                  inf_dominant_topic_t2['dominant_topic'].value_counts(0), inf_dominant_topic_t2['dominant_topic'].value_counts(1)], axis=1)
temp.columns = ['doc_count_all','doc_pct_all','doc_count_t1','doc_pct_t1','doc_count_t2','doc_pct_t2']
temp['doc_count_diff'] = temp['doc_count_t2'] - temp['doc_count_t1']
temp['doc_pct_diff'] = ((temp['doc_pct_t2'] - temp['doc_pct_t1'])/temp['doc_pct_t1'])*100

inf_topic_freq = inf_dominant_topic_rep[['dominant_topic','topic_keywords']].merge(temp, how='inner',left_index=True,right_index=True)
inf_topic_freq.to_csv(INF_PATH + 'dominantTopic_freq_' + INF_REV_SRC + '_' + str(datetime.datetime.now().date()).replace('-','') + '.csv')

In [ ]:
print(len(inf_topic_freq))
inf_topic_freq.head(20)

#### Topic Prevalence

In [ ]:
inf_topic_prev = pd.concat([inf_doc_topic.mean(),inf_doc_topic_t1.mean(),inf_doc_topic_t2.mean()], axis=1)
inf_topic_prev.columns = ['topic_prev_all','topic_prev_t1','topic_prev_t2']
inf_topic_prev['topic_prev_pct_diff'] = ((inf_topic_prev['topic_prev_t2'] - inf_topic_prev['topic_prev_t1'])/inf_topic_prev['topic_prev_t1'])*100
inf_topic_prev.to_csv(INF_PATH + 'topic_prev_' + INF_REV_SRC + '_' + str(datetime.datetime.now().date()).replace('-','') + '.csv')

In [ ]:
print(len(inf_topic_prev))
inf_topic_prev.head(20)

#### Word Count Histogram

In [ ]:
doc_lens = [len(d.split()) for d in inf_dominant_topic.doc_text]

# Plot
fig,ax = plt.subplots(1,1,figsize=(16,8))
ax.hist(doc_lens, bins = 200, color='navy')
ax.text(max(doc_lens)/2,HEIGHTS['small'][4],"Mean: " + str(round(np.mean(doc_lens))), fontsize=14)
ax.text(max(doc_lens)/2,HEIGHTS['small'][3],"Median: " + str(round(np.median(doc_lens))), fontsize=14)
ax.text(max(doc_lens)/2,HEIGHTS['small'][2],"Stdev: " + str(round(np.std(doc_lens))), fontsize=14)
ax.text(max(doc_lens)/2,HEIGHTS['small'][1],"1%: " + str(round(np.quantile(doc_lens, q=0.01))), fontsize=14)
ax.text(max(doc_lens)/2,HEIGHTS['small'][0],"99%: " + str(round(np.quantile(doc_lens, q=0.99))), fontsize=14)

ax.set_xlim=(0, max(doc_lens))
ax.set_ylabel('Number of Documents')
ax.set_xlabel('Document Word Count')
ax.tick_params(size=16)
ax.set_xticks(np.linspace(0,max(doc_lens),9))
ax.set_title('Distribution of Document Word Counts', fontdict=dict(size=22))
#plt.show()

name = INF_PATH + 'wordCount_hist_' + INF_REV_SRC + '_' + str(datetime.datetime.now().date()).replace('-','') + '.png'
plt.savefig(name,bbox_inches='tight')

#### Word Count Histogram by Topic

In [ ]:
import seaborn as sns
import matplotlib.colors as mcolors
cols = [color for name, color in mcolors.XKCD_COLORS.items()]  # more colors: 'mcolors.XKCD_COLORS'

fig, axes = plt.subplots(5,NUM_COLS[NUM_TOPICS],figsize=(16,20), sharex=True, sharey=True)

for i, ax in enumerate(axes.flatten()):    
    inf_dominant_topic_sub = inf_dominant_topic.loc[inf_dominant_topic.dominant_topic == i, :]
    doc_lens = [len(d.split()) for d in inf_dominant_topic_sub.doc_text]
    ax.hist(doc_lens, bins = 50, color=cols[i])
    ax.tick_params(axis='y', labelcolor=cols[i], color=cols[i])
    sns.kdeplot(doc_lens, color="black", shade=False, ax=ax.twinx())
    ax.set(xlim=(0, 1000), xlabel='Document Word Count')
    ax.set_ylabel('Number of Documents', color=cols[i])
    ax.set_title('Topic: '+str(i), fontdict=dict(size=16, color=cols[i]))

fig.tight_layout()
fig.subplots_adjust(top=0.90)
plt.xticks(np.linspace(0,1000,9))
fig.suptitle('Distribution of Document Word Counts by Dominant Topic', fontsize=22)
plt.show()

name = INF_PATH + 'wordCount_hist_topics_' + INF_REV_SRC + '_' + str(datetime.datetime.now().date()).replace('-','') + '.png'
fig.savefig(name,bbox_inches='tight')

#### Word Counts by Topic

In [ ]:
from collections import Counter
topics = ldamallet.show_topics(formatted=False,num_topics=NUM_TOPICS)
data_flat = [w for w_list in inf_lemma for w in w_list]
counter = Counter(data_flat)

out = []
for i, topic in topics:
    for word, weight in topic:
        out.append([word, i , weight, counter[word]])

df = pd.DataFrame(out, columns=['word', 'topic_id', 'importance', 'word_count'])        

# Plot Word Count and Weights of Topic Keywords
fig, axes = plt.subplots(5,NUM_COLS[NUM_TOPICS], figsize=(20,20), sharey=True)
cols = [color for name, color in mcolors.XKCD_COLORS.items()]
for i, ax in enumerate(axes.flatten()):
    ax.bar(x='word', height="word_count", data=df.loc[df.topic_id==i, :], color=cols[i], width=0.5, alpha=0.4, label='Word Count')
    ax_twin = ax.twinx()
    ax_twin.bar(x='word', height="importance", data=df.loc[df.topic_id==i, :], color=cols[i], width=0.2, label='Weights')
    ax.set_ylabel('Word Count', color=cols[i])
    ax_twin.set_ylim(0, 0.060); ax.set_ylim(0, 2000)
    ax.set_title('Topic: ' + str(i), color=cols[i], fontsize=16)
    ax.tick_params(axis='y', left=False)
    ax.set_xticklabels(df.loc[df.topic_id==i, 'word'], rotation=30, horizontalalignment= 'right')
    ax.legend(loc='upper left'); ax_twin.legend(loc='upper right')

fig.tight_layout(w_pad=2)    
fig.suptitle('Word Count and Importance of Topic Keywords', fontsize=22, y=1.05)    
plt.show()

name = INF_PATH + 'wordCount_terms_topics_' + INF_REV_SRC + '_' + str(datetime.datetime.now().date()).replace('-','') + '.png'
fig.savefig(name,bbox_inches='tight')

#### Topic Clusters

In [ ]:
# Get topic weights and dominant topics ------------
from sklearn.manifold import TSNE
from bokeh.plotting import figure, output_file, show
from bokeh.models import Label
from bokeh.io import output_notebook

# Get topic weights
topic_weights = []
for i, row_list in enumerate(ldamallet[inf_corpus]):
    topic_weights.append([w for i, w in row_list])

# Array of topic weights    
arr = pd.DataFrame(topic_weights).fillna(0).values

# Keep the well separated points (optional)
arr = arr[np.amax(arr, axis=1) > 0.1]

# Dominant topic number in each doc
topic_num = np.argmax(arr, axis=1)

# tSNE Dimension Reduction
tsne_model = TSNE(n_components=2, verbose=1, random_state=0, angle=.99, init='pca')
tsne_lda = tsne_model.fit_transform(arr)

# Plot the Topic Clusters using Bokeh
output_notebook()
n_topics = NUM_TOPICS
mycolors = np.array([color for name, color in mcolors.XKCD_COLORS.items()])
plot = figure(title="t-SNE Clustering of {} LDA Topics".format(n_topics), 
              plot_width=900, plot_height=700)
plot.scatter(x=tsne_lda[:,0], y=tsne_lda[:,1], color=mycolors[topic_num])
show(plot)

name = INF_PATH + 'doc_topic_clusters_' + INF_REV_SRC + '_' + str(datetime.datetime.now().date()).replace('-','') + '.png'
export_png(plot, filename=name)

## Experiments

### Time-Series

In [ ]:
if REV_SRC == 'all':
    ts_data = pd.concat([all_data_t1,all_data_t2]).reset_index()
elif REV_SRC == 'covid':
    ts_data = pd.concat([covid_data_t1,covid_data_t2]).reset_index()
doc_topic_ts = pd.concat([doc_topic,ts_data[['rev_created_time']]],axis=1).sort_values('rev_created_time')

In [ ]:
ax = doc_topic_ts.groupby(['rev_created_time']).mean().plot(figsize=(18,12))
ax.set_ylim(0.01,0.15)

### Ratings and Correlation

In [ ]:
doc_topic_rating = pd.concat([doc_topic,ts_data[['rev_created_time','rev_rating']]],axis=1).sort_values('rev_created_time')

In [ ]:
doc_topic_rating.groupby(['rev_created_time']).mean().head()

In [ ]:
doc_topic_rating.groupby(['rev_created_time']).mean().corr()

In [ ]:
ax = doc_topic_rating.groupby(['rev_created_time']).mean()['rev_rating'].plot(figsize=(18,12))
#ax.set_ylim(1.5,3.5)

In [ ]:
ax = doc_topic_rating.groupby(['rev_created_time']).mean().rolling(window='7d').mean()['rev_rating'].plot(figsize=(18,12))
#ax.set_ylim(1.5,3.5)

## Unused / Deprecated Code

In [ ]:
"""

# Get covid reviews
start_date = pd.to_datetime('2019-12-01')
end_date = pd.to_datetime(date_str_fmt)
date_range = pd.date_range(start=start_date, end=end_date)

# Baseline data
reviews_filtered = reviews[reviews['rev_created_time'].isin(date_range)]
reviews_fac = facilities[facilities['fac_id'].isin(reviews_filtered['fac_id'])]
reviews_cat = categories[categories['fac_id'].isin(reviews_fac['fac_id'])]
reviews_data = reviews_filtered.merge(reviews_fac, how='inner', on='fac_id')

# Get covid facilities and categories and merge
covid_reviews = reviews_filtered[reviews_filtered['review'].str.contains(SEARCH_STRING, regex=True)]
covid_fac = facilities[facilities['fac_id'].isin(covid_reviews['fac_id'])]
covid_cat = categories[categories['fac_id'].isin(covid_fac['fac_id'])]
covid_data = covid_reviews.merge(covid_fac, how='inner', on='fac_id')

# State dictionaries
states = covid_data['state'].unique()
state_data = dict()
cat_data = dict()
for state in states:
    state_data[state] = covid_data[covid_data['state']==state]
    cat_data[state] = covid_cat[covid_cat['fac_id'].isin(state_data[state]['fac_id'])]

# Save data
#reviews_data.to_csv(DATA_PATH + 'reviews_data_' + str(datetime.datetime.now().date()).replace('-','') + '.csv')
#covid_data.to_csv(DATA_PATH + 'covid_data_' + str(datetime.datetime.now().date()).replace('-','') + '.csv')

"""

In [ ]:
"""

state = None
start_date = pd.to_datetime('2019-12-01')
end_date = pd.to_datetime(date_str_fmt)
plot_data = get_time_series(covid_data, start_date, end_date, state)

ax = plot_data.plot(figsize=(16,8))
if state==None:
    state_txt = 'United States'
else:
    state_txt = state
ax.set_title('Number of COVID-19 Yelp Reviews Over Time: '+state_txt, fontsize=16)
ax.set_xlabel('Date',fontsize=14)
ax.set_ylabel('Review Count',fontsize=14)

"""

In [ ]:
"""

def compute_coherence_values(dictionary, corpus, texts, limit, start, step):
    coherence_values = []
    model_list = []
    for num_topics in range(start, limit, step):
        model = gensim.models.wrappers.LdaMallet(MALLET_PATH, corpus=corpus, num_topics=num_topics, id2word=id2word)
        model_list.append(model)
        coherencemodel = CoherenceModel(model=model, texts=texts, dictionary=dictionary, coherence='c_v')
        coherence_values.append(coherencemodel.get_coherence())

    return model_list, coherence_values

"""

In [ ]:
"""

# Generate model output
start,limit,step=5,26,5
model_list,coherence_values=compute_coherence_values(dictionary=id2word,corpus=corpus,texts=data_lemmatized,start=start,limit=limit,step=step)

# Plot coherence
x = range(start, limit, step)
plt.plot(x, coherence_values)
plt.xlabel("Num Topics")
plt.ylabel("Coherence score")
plt.legend(("coherence_values"), loc='best')
plt.show()

# Print coherence scores
for m, cv in zip(x, coherence_values):
    print("Num Topics =", m, " has Coherence Value of", round(cv, 4))

"""

In [ ]:
"""

#### Gensim Model ####

# Gensim topic model
ldagensim = gensim.models.ldamodel.LdaModel(corpus=corpus,id2word=id2word,num_topics=NUM_TOPICS,random_state=100,
                                            update_every=1,chunksize=100,passes=10,alpha='auto',per_word_topics=True)
# Show Topics
pprint(ldagensim.show_topics(num_topics=NUM_TOPICS,num_words=WORD_COUNT,formatted=False))
doc_ldagensim = ldagensim[corpus]

# Compute Coherence Score
coherence_model_ldagensim = CoherenceModel(model=ldagensim, texts=data_lemmatized, dictionary=id2word, coherence='c_v')
coherence_ldagensim = coherence_model_ldagensim.get_coherence()
print('\nCoherence Score: ', coherence_ldagensim)
print('\nPerplexity: ', ldagensim.log_perplexity(corpus))

# Topic Output
topic_term_ldagensim_top20 = get_topic_terms(ldagensim)
topic_term_ldagensim_top20.round(3).head(5)

# Visualize the topics
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(ldagensim, corpus, id2word)
vis

"""

In [ ]:
"""

topic_term_temp = pd.DataFrame(ldamalletConvert.get_topics())
topic_term_temp.columns = [id2word[item] for item in topic_term_temp.columns]
topic_term_temp.loc[0].sort_values(ascending=False)

"""

In [ ]:
"""

print('2020-01-01 to 2020-03-15')
print('All reviews:',len(all_data_t1))
print('Covid reviews:',len(covid_data_t1))
print('Telemedicine reviews:',len(tele_data_t1))
print('Covid and Telemedicine reviews:',len(merge_data_t1))
print()
print('2020-03-16 to 2020-06-10')
print('All reviews:',len(all_data_t2))
print('Covid reviews:',len(covid_data_t2))
print('Telemedicine reviews:',len(tele_data_t2))
print('Covid and Telemedicine reviews:',len(merge_data_t2))

"""

In [ ]:
"""

# Select raw data
INF_REV_SRC = 'tele'
inf_data_t1 = tele_data_t1
inf_data_t2 = tele_data_t2
inf_data = pd.concat([inf_data_t1,inf_data_t2]).review.values.tolist()

"""

In [ ]:
"""

# Sentence Coloring of N Sentences
from matplotlib.patches import Rectangle

def sentences_chart(lda_model=ldamallet, corpus=corpus, start = 0, end = 5):
    corp = corpus[start:end]
    mycolors = [color for name, color in mcolors.XKCD_COLORS.items()]

    fig, axes = plt.subplots(end-start, 1, figsize=(20, (end-start)*0.95), dpi=160)       
    axes[0].axis('off')
    for i, ax in enumerate(axes):
        if i > 0:
            corp_cur = corp[i-1] 
            topic_percs, wordid_topics = lda_model[corp_cur]
            word_dominanttopic = [(lda_model.id2word[wd], topic[0]) for wd, topic in wordid_topics]    
            ax.text(0.01, 0.5, "Doc " + str(i-1) + ": ", verticalalignment='center',
                    fontsize=16, color='black', transform=ax.transAxes, fontweight=700)

            # Draw Rectange
            topic_percs_sorted = sorted(topic_percs, key=lambda x: (x[1]), reverse=True)
            ax.add_patch(Rectangle((0.0, 0.05), 0.99, 0.90, fill=None, alpha=1, 
                                   color=mycolors[topic_percs_sorted[0][0]], linewidth=2))

            word_pos = 0.06
            for j, (word, topics) in enumerate(word_dominanttopic):
                if j < 14:
                    ax.text(word_pos, 0.5, word,
                            horizontalalignment='left',
                            verticalalignment='center',
                            fontsize=16, color=mycolors[topics],
                            transform=ax.transAxes, fontweight=700)
                    word_pos += .009 * len(word)  # to move the word for the next iter
                    ax.axis('off')
            ax.text(word_pos, 0.5, '. . .',
                    horizontalalignment='left',
                    verticalalignment='center',
                    fontsize=16, color='black',
                    transform=ax.transAxes)       

    plt.subplots_adjust(wspace=0, hspace=0)
    plt.suptitle('Sentence Topic Coloring for Documents: ' + str(start) + ' to ' + str(end-2), fontsize=22, y=0.95, fontweight=700)
    plt.tight_layout()
    plt.show()

sentences_chart()

"""